# Preprocesamiento de Datos y Feature Engineering

## 🎯 Objetivo

En este notebook aprenderemos las técnicas **más importantes** para preparar datos antes de entrenar modelos de Machine Learning.

### 📌 Tema Central

> **"Los modelos son tan buenos como los datos que reciben."**

El 80% del trabajo en un proyecto de ML es **preparación de datos**. Un modelo mediocre con datos bien preparados supera a un modelo sofisticado con datos sucios.

---

## 📚 Contenido

### Parte 1: Preprocesamiento
1. Valores Faltantes
2. Outliers
3. Duplicados

### Parte 2: Transformaciones
4. Encoding de Variables Categóricas
5. Escalamiento
6. Normalización

### Parte 3: Feature Engineering
7. Creación de Features
8. Transformaciones no lineales
9. Interacciones

### Parte 4: Selección de Features
10. Métodos Filter
11. Métodos Wrapper
12. Métodos Embedded

## Parte 1: Limpieza de Datos

### 🧹 1.1 Valores Faltantes (Missing Values)

Los valores faltantes son uno de los problemas más comunes en datos reales.

#### Estrategias:

| Método | Cuándo usarlo | Ventajas | Desventajas |
|--------|----------------|----------|-------------|
| **Eliminar filas** | <5% missing | Simple | Pierde información |
| **Imputar con media/mediana** | Numéricos | Preserva tamaño | Introduce bias |
| **Imputar con moda** | Categóricos | Simple | Puede distorsionar distribución |
| **Imputar con modelo** | >20% missing | Preciso | Computacionalmente costoso |
| **Indicador missing** | Información valiosa | Captura patrón | Aumenta dimensionalidad |

#### 💡 Tips:

* **Analizar patrón**: ¿Missing aleatorio o sistemático?
* **No eliminar automáticamente**: Valores faltantes pueden tener significado
* **Documentar decisiones**: Explicar qué método y por qué

In [0]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

# Crear dataset con valores faltantes
np.random.seed(42)
df = pd.DataFrame({
    'edad': [25, 30, np.nan, 45, np.nan, 35, 50, np.nan],
    'salario': [50000, 60000, np.nan, 80000, 70000, np.nan, 90000, 75000],
    'ciudad': ['NY', 'LA', np.nan, 'CHI', 'NY', 'LA', np.nan, 'CHI'],
    'compro': [1, 0, 1, 1, 0, np.nan, 1, 0]
})

print("┌──────────────────────────────────────────────────┐")
print("│        DATASET ORIGINAL CON VALORES FALTANTES        │")
print("└──────────────────────────────────────────────────┘")
print(df)
print(f"\n📉 Missing values por columna:")
print(df.isnull().sum())

# Método 1: Imputar con media/mediana
imputer_num = SimpleImputer(strategy='median')
df_imputed = df.copy()
df_imputed[['edad', 'salario']] = imputer_num.fit_transform(df[['edad', 'salario']])

print("\n\n✅ DATASET DESPUÉS DE IMPUTACIÓN (media/mediana):")
print(df_imputed)

## 1.2 Outliers (Valores Atípicos)

### 🔍 Definición

Outliers son observaciones que se desvían significativamente del resto de los datos.

### Métodos de Detección:

#### 1️⃣ **Método IQR (Interquartile Range)**

$$\text{IQR} = Q_3 - Q_1$$

$$\text{Lower Bound} = Q_1 - 1.5 \times \text{IQR}$$
$$\text{Upper Bound} = Q_3 + 1.5 \times \text{IQR}$$

#### 2️⃣ **Método Z-Score**

$$z = \frac{x - \mu}{\sigma}$$

Outliers: $|z| > 3$

#### 3️⃣ **Isolation Forest**

Algoritmo de ML para detección de anomalías.

### ⚠️ ¿Qué hacer con outliers?

* **NO eliminar automáticamente** - pueden ser valores válidos
* **Investigar**: ¿Error de medición o fenómeno real?
* **Opciones**: Eliminar, transformar (log, winsorize), usar modelos robustos

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Crear datos con outliers
np.random.seed(42)
data_normal = np.random.normal(100, 15, 95)
outliers = np.array([200, 210, 220, 5, 10])  # Outliers
data = np.concatenate([data_normal, outliers])
df_outliers = pd.DataFrame({'valor': data})

print("┌──────────────────────────────────────────────────┐")
print("│         DETECCIÓN DE OUTLIERS - MÉTODO IQR           │")
print("└──────────────────────────────────────────────────┘")

# Método IQR
Q1 = df_outliers['valor'].quantile(0.25)
Q3 = df_outliers['valor'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1: {Q1:.2f}")
print(f"Q3: {Q3:.2f}")
print(f"IQR: {IQR:.2f}")
print(f"Lower Bound: {lower_bound:.2f}")
print(f"Upper Bound: {upper_bound:.2f}")

outliers_mask = (df_outliers['valor'] < lower_bound) | (df_outliers['valor'] > upper_bound)
print(f"\n🚨 Outliers detectados: {outliers_mask.sum()}")
print(df_outliers[outliers_mask])

## Parte 2: Transformaciones de Datos

### 2.1 Encoding de Variables Categóricas

Los modelos de ML requieren inputs numéricos. Debemos convertir variables categóricas.

| Método | Uso | Ejemplo | Ventajas | Desventajas |
|--------|-----|---------|----------|-------------|
| **Label Encoding** | Ordinales | Tamaño: Pequeño(0), Mediano(1), Grande(2) | Simple | Implica orden |
| **One-Hot Encoding** | Nominales | Color: Rojo[1,0,0], Verde[0,1,0], Azul[0,0,1] | No asume orden | Alta dimensionalidad |
| **Target Encoding** | Alta cardinalidad | Promedio del target por categoría | Reduce dimensionalidad | Puede causar overfitting |

### 2.2 Escalamiento (Scaling)

#### **StandardScaler (Z-score):**

$$x' = \frac{x - \mu}{\sigma}$$

* Media = 0, Desviación estándar = 1
* Usa cuando: Distribución normal, presencia de outliers moderados

#### **MinMaxScaler:**

$$x' = \frac{x - x_{min}}{x_{max} - x_{min}}$$

* Rango [0, 1]
* Usa cuando: Necesitas valores acotados, redes neuronales

#### **RobustScaler:**

Usa mediana e IQR (robusto a outliers)

* Usa cuando: Muchos outliers

In [0]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler

# Dataset de ejemplo
df_transform = pd.DataFrame({
    'ciudad': ['NY', 'LA', 'CHI', 'NY', 'LA', 'CHI'],
    'educacion': ['Secundaria', 'Universidad', 'Primaria', 'Universidad', 'Secundaria', 'Universidad'],
    'edad': [25, 45, 30, 50, 28, 35],
    'salario': [50000, 90000, 60000, 95000, 55000, 70000]
})

print("📊 ENCODING - ONE-HOT:")
df_encoded = pd.get_dummies(df_transform, columns=['ciudad'], prefix='ciudad')
print(df_encoded.head())

print("\n📊 ESCALAMIENTO:")
# StandardScaler
scaler_standard = StandardScaler()
df_transform['edad_standard'] = scaler_standard.fit_transform(df_transform[['edad']])

# MinMaxScaler
scaler_minmax = MinMaxScaler()
df_transform['salario_minmax'] = scaler_minmax.fit_transform(df_transform[['salario']])

print(df_transform[['edad', 'edad_standard', 'salario', 'salario_minmax']])

## Parte 3: Feature Engineering

### 🎯 Definición

> **Feature Engineering** es el arte de crear nuevas features a partir de datos existentes para mejorar el performance del modelo.

### 💡 Regla de Oro

**"Features beats algorithms."** - Una feature bien diseñada puede mejorar más el modelo que cambiar de algoritmo.

### Técnicas Comunes:

#### 1️⃣ **Features Polinomiales**

* $x^2, x^3, \sqrt{x}$
* Capturan relaciones no lineales

#### 2️⃣ **Interacciones**

* $x_1 \times x_2$
* Capturan relaciones entre features

#### 3️⃣ **Agregaciones**

* Suma, promedio, max, min por grupo
* Útil en series temporales

#### 4️⃣ **Binning (Discretización)**

* Convertir continuo a categórico
* Edad → Grupos: Joven, Adulto, Mayor

#### 5️⃣ **Fechas**

* Extraer: año, mes, día, día_semana
* Features cíclicas: $\sin(\text{mes}), \cos(\text{mes})$

#### 6️⃣ **Texto**

* Longitud, cantidad de palabras, sentimiento
* TF-IDF, embeddings

In [0]:
# Dataset de ejemplo: Transacciones de e-commerce
df_fe = pd.DataFrame({
    'fecha_compra': pd.to_datetime(['2024-01-15', '2024-02-20', '2024-03-10', '2024-01-22', '2024-02-14']),
    'precio': [50, 150, 30, 200, 75],
    'cantidad': [2, 1, 5, 1, 3],
    'categoria': ['Electrónica', 'Ropa', 'Hogar', 'Electrónica', 'Hogar']
})

print("🔧 FEATURE ENGINEERING - CREACIÓN DE FEATURES:\n")

# 1. Feature de interacción
df_fe['total_compra'] = df_fe['precio'] * df_fe['cantidad']
print("1. Total compra (precio × cantidad):")
print(df_fe[['precio', 'cantidad', 'total_compra']].head())

# 2. Features de fecha
df_fe['mes'] = df_fe['fecha_compra'].dt.month
df_fe['dia_semana'] = df_fe['fecha_compra'].dt.dayofweek
df_fe['trimestre'] = df_fe['fecha_compra'].dt.quarter
print("\n2. Features de fecha:")
print(df_fe[['fecha_compra', 'mes', 'dia_semana', 'trimestre']].head())

# 3. Binning
df_fe['rango_precio'] = pd.cut(df_fe['precio'], bins=[0, 50, 100, 200], labels=['Bajo', 'Medio', 'Alto'])
print("\n3. Binning de precio:")
print(df_fe[['precio', 'rango_precio']].head())

## Parte 4: Selección de Features

### ⚠️ El Problema de Muchas Features

**"Curse of Dimensionality"** - Demasiadas features:
* ❌ Overfitting
* ❌ Lentitud
* ❌ Dificultad de interpretación

### Métodos de Selección:

#### 1️⃣ **Filter Methods**

* Basados en estadísticas (correlación, chi-cuadrado, mutual information)
* Rápidos, independientes del modelo
* Ejemplos: `SelectKBest`, `VarianceThreshold`

#### 2️⃣ **Wrapper Methods**

* Evalúan subconjuntos de features entrenando modelo
* Lentos pero precisos
* Ejemplos: RFE (Recursive Feature Elimination)

#### 3️⃣ **Embedded Methods**

* Selección durante el entrenamiento
* Balance velocidad-precisión
* Ejemplos: Lasso (L1), Random Forest feature importance

### 🎯 Estrategia Recomendada:

1. **Eliminar features de baja varianza**
2. **Eliminar features altamente correlacionadas**
3. **Aplicar método embedded** (ej: Random Forest importance)
4. **Validar con cross-validation**

In [0]:
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
import seaborn as sns
import matplotlib.pyplot as plt

# Crear dataset para clasificación
np.random.seed(42)
X = np.random.randn(100, 10)
y = (X[:, 0] + X[:, 1] > 0).astype(int)  # Target depende solo de 2 features

print("🎯 SELECCIÓN DE FEATURES:\n")

# 1. VarianceThreshold - eliminar features con baja varianza
selector_var = VarianceThreshold(threshold=0.1)
X_var = selector_var.fit_transform(X)
print(f"1. VarianceThreshold: {X.shape[1]} → {X_var.shape[1]} features")

# 2. SelectKBest - top k features por correlación con target
selector_k = SelectKBest(f_classif, k=5)
X_k = selector_k.fit_transform(X, y)
print(f"\n2. SelectKBest (k=5): Top 5 features")
scores = selector_k.scores_
for i, score in enumerate(scores):
    print(f"   Feature {i}: {score:.2f}")

# 3. Random Forest feature importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)
importances = rf.feature_importances_
print(f"\n3. Random Forest - Feature Importance:")
for i, imp in enumerate(importances):
    print(f"   Feature {i}: {imp:.3f}")

## 📍d Conclusiones y Mejores Prácticas

### 🏆 Key Takeaways

1. **El 80% del trabajo en ML es preparación de datos** - Invértele el tiempo necesario

2. **No hay una estrategia universal** - Cada dataset es diferente

3. **Explora antes de transformar** - Entiende tus datos primero (EDA)

4. **Documenta decisiones** - Por qué imputaste con media, por qué eliminaste outliers

5. **Usa pipelines** - Sklearn Pipelines para reproducibilidad

6. **Evita data leakage** - Fit solo en training, transform en test

### 🚨 Errores Comunes:

* ❌ Escalar antes de train/test split
* ❌ Eliminar outliers sin investigar
* ❌ Crear features usando información del futuro
* ❌ No validar features con cross-validation
* ❌ Sobre-ingeniería: crear demasiadas features sin validar

### ✅ Checklist de Preprocesamiento:

- [ ] Valores faltantes manejados
- [ ] Outliers analizados (eliminar/transformar/mantener)
- [ ] Variables categóricas codificadas
- [ ] Features numéricas escaladas
- [ ] Features creadas y validadas
- [ ] Features seleccionadas
- [ ] Pipeline documentado y reproducible

### 📚 Recursos:

* **Libro**: "Feature Engineering for Machine Learning" (Alice Zheng)
* **Práctica**: Kaggle competitions - estudiar kernels ganadores
* **Herramientas**: scikit-learn, pandas, category_encoders

In [0]:
# DBTITLE 1,## Parte 2: Transformaciones# MAGIC %md# MAGIC ## Parte 2: Transformaciones de Datos# MAGIC# MAGIC ### 2.1 Encoding de Variables Categóricas# MAGIC# MAGIC Los modelos de ML requieren inputs numéricos. Debemos convertir variables categóricas.# MAGIC# MAGIC | Método | Uso | Ejemplo | Ventajas | Desventajas |# MAGIC |--------|-----|---------|----------|-------------|# MAGIC | **Label Encoding** | Ordinales | Educación: Primaria=1, Secundaria=2, Universidad=3 | Simple, compacto | Implica orden |# MAGIC | **One-Hot Encoding** | Nominales de baja cardinalidad | Ciudad: NY, LA, CHI → 3 columnas binarias | Sin orden implícito | Muchas columnas |# MAGIC | **Target Encoding** | Alta cardinalidad | Reemplazar por media del target | Compacto | Riesgo de overfitting |# MAGIC | **Frequency Encoding** | Alta cardinalidad | Reemplazar por frecuencia | Simple | Pierde identidad |# MAGIC# MAGIC ### 2.2 Scaling y Normalización# MAGIC# MAGIC Algunos algoritmos (KNN, SVM, redes neuronales) son sensibles a la escala de features.# MAGIC# MAGIC #### Standardization (Z-Score Normalization)# MAGIC# MAGIC $$x' = \frac{x - \mu}{\sigma}$$# MAGIC# MAGIC * Media = 0, Desviación estándar = 1# MAGIC * **Usar cuando**: Features siguen distribución normal# MAGIC * **Algoritmos**: SVM, Redes Neuronales, PCA# MAGIC# MAGIC #### Min-Max Normalization# MAGIC# MAGIC $$x' = \frac{x - \min(x)}{\max(x) - \min(x)}$$# MAGIC# MAGIC * Rango [0, 1]# MAGIC * **Usar cuando**: Quieres un rango específico# MAGIC * **Sensible a outliers**# MAGIC# MAGIC #### Robust Scaling# MAGIC# MAGIC $$x' = \frac{x - \text{median}(x)}{\text{IQR}}$$# MAGIC# MAGIC * Robusto a outliers# MAGIC * Usa mediana y IQR# MAGIC * **Usar cuando**: Hay outliers

In [0]:
# DBTITLE 1,Ejemplo: Encoding y Scalingfrom sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler# Dataset de ejemplodf_transform = pd.DataFrame({    'ciudad': ['NY', 'LA', 'CHI', 'NY', 'LA', 'CHI'],    'educacion': ['Secundaria', 'Universidad', 'Primaria', 'Universidad', 'Secundaria', 'Universidad'],    'edad': [25, 35, 22, 40, 28, 38],    'salario': [50000, 75000, 35000, 85000, 55000, 80000]})print("┌──────────────────────────────────────────────────┐")print("│          ENCODING Y SCALING - EJEMPLOS            │")print("└──────────────────────────────────────────────────┘")print("\n📊 Dataset original:")print(df_transform)# 1. Label Encoding (para ordinales)print("\n" + "="*50)print("🔧 1. LABEL ENCODING (Educación - Ordinal)")print("="*50)educacion_mapping = {'Primaria': 1, 'Secundaria': 2, 'Universidad': 3}df_transform['educacion_encoded'] = df_transform['educacion'].map(educacion_mapping)print("\n✅ Resultado:")print(df_transform[['educacion', 'educacion_encoded']])# 2. One-Hot Encoding (para nominales)print("\n" + "="*50)print("🔧 2. ONE-HOT ENCODING (Ciudad - Nominal)")print("="*50)df_onehot = pd.get_dummies(df_transform['ciudad'], prefix='ciudad')print("\n✅ Resultado:")print(df_onehot)# 3. Standardizationprint("\n" + "="*50)print("🔧 3. STANDARDIZATION (Z-Score)")print("="*50)scaler_std = StandardScaler()df_transform['salario_std'] = scaler_std.fit_transform(df_transform[['salario']])print("\n✅ Resultado (Salario):")print(df_transform[['salario', 'salario_std']])print(f"\nMedia: {df_transform['salario_std'].mean():.10f}")print(f"Std: {df_transform['salario_std'].std():.2f}")# 4. Min-Max Normalizationprint("\n" + "="*50)print("🔧 4. MIN-MAX NORMALIZATION")print("="*50)scaler_minmax = MinMaxScaler()df_transform['edad_minmax'] = scaler_minmax.fit_transform(df_transform[['edad']])print("\n✅ Resultado (Edad):")print(df_transform[['edad', 'edad_minmax']])print(f"\nMin: {df_transform['edad_minmax'].min()}")print(f"Max: {df_transform['edad_minmax'].max()}")# Comparación finalprint("\n" + "="*50)print("📊 DATASET TRANSFORMADO COMPLETO")print("="*50)print("\n")print(df_transform)

In [0]:
# DBTITLE 1,## Parte 3: Feature Engineering# MAGIC %md# MAGIC ## Parte 3: Feature Engineering# MAGIC# MAGIC ### 🎯 Definición# MAGIC# MAGIC > **Feature Engineering** es el arte de crear nuevas features a partir de datos existentes para mejorar el performance del modelo.# MAGIC# MAGIC ### 💡 Regla de Oro# MAGIC# MAGIC **"Features beat algorithms"** - Mejores features tienen más impacto que mejores algoritmos.# MAGIC# MAGIC ### Tipos de Features# MAGIC# MAGIC #### 1️⃣ **Features de Fecha/Tiempo**# MAGIC# MAGIC De una columna `fecha`:# MAGIC * Año, mes, día# MAGIC * Día de la semana, fin de semana# MAGIC * Trimestre, semestre# MAGIC * Hora, minuto (para timestamps)# MAGIC * Días desde evento# MAGIC# MAGIC #### 2️⃣ **Interacciones**# MAGIC# MAGIC Combinar dos o más features:# MAGIC * Producto: `precio * cantidad = gasto_total`# MAGIC * Ratio: `ingresos / gastos = ratio_ahorro`# MAGIC * Diferencia: `fecha_entrega - fecha_pedido = dias_envio`# MAGIC# MAGIC #### 3️⃣ **Agregaciones**# MAGIC# MAGIC Por grupos:# MAGIC * `promedio_compra_por_cliente`# MAGIC * `total_ventas_por_ciudad`# MAGIC * `max_precio_por_categoria`# MAGIC# MAGIC #### 4️⃣ **Binning**# MAGIC# MAGIC Convertir continuas en categóricas:# MAGIC * Edad → Grupos etáreos: 0-18, 19-35, 36-65, 65+# MAGIC * Salario → Rangos: Bajo, Medio, Alto# MAGIC# MAGIC #### 5️⃣ **Transformaciones Matemáticas**# MAGIC# MAGIC * **Log**: $\log(x)$ - Para distribución sesgada# MAGIC * **Sqrt**: $\sqrt{x}$ - Reduce impacto de valores grandes# MAGIC * **Polynomial**: $x^2, x^3$ - Captura relaciones no-lineales

In [0]:
# DBTITLE 1,Ejemplo: Feature Engineering completo# Dataset de ejemplo: Transacciones de e-commercedf_fe = pd.DataFrame({    'fecha_compra': pd.to_datetime(['2024-01-15', '2024-02-20', '2024-03-10', '2024-01-22', '2024-02-14']),    'precio': [50, 150, 30, 200, 75],    'cantidad': [2, 1, 5, 1, 3],    'cliente_id': ['C1', 'C2', 'C1', 'C3', 'C1'],    'categoria': ['Electrónica', 'Ropa', 'Libros', 'Electrónica', 'Libros'],    'edad_cliente': [25, 45, 25, 60, 25]})print("┌──────────────────────────────────────────────────┐")print("│            FEATURE ENGINEERING - EJEMPLOS          │")print("└──────────────────────────────────────────────────┘")print("\n📊 Dataset original:")print(df_fe)# 1. Features de fecha/tiempoprint("\n" + "="*50)print("🔧 1. FEATURES DE FECHA/TIEMPO")print("="*50)df_fe['mes'] = df_fe['fecha_compra'].dt.monthdf_fe['dia_semana'] = df_fe['fecha_compra'].dt.dayofweekdf_fe['es_fin_semana'] = (df_fe['dia_semana'] >= 5).astype(int)df_fe['trimestre'] = df_fe['fecha_compra'].dt.quarterprint("\n✅ Features creadas:")print(df_fe[['fecha_compra', 'mes', 'dia_semana', 'es_fin_semana', 'trimestre']])# 2. Interacciones (Features derivadas)print("\n" + "="*50)print("🔧 2. INTERACCIONES")print("="*50)df_fe['gasto_total'] = df_fe['precio'] * df_fe['cantidad']df_fe['precio_por_unidad'] = df_fe['precio']  # Ya está, pero podría ser gasto_total / cantidadprint("\n✅ Features de interacción:")print(df_fe[['precio', 'cantidad', 'gasto_total']])# 3. Agregaciones por clienteprint("\n" + "="*50)print("🔧 3. AGREGACIONES POR CLIENTE")print("="*50)agg_cliente = df_fe.groupby('cliente_id').agg({    'gasto_total': ['sum', 'mean', 'count'],    'cantidad': 'sum'}).reset_index()agg_cliente.columns = ['cliente_id', 'total_gastado', 'gasto_promedio', 'num_compras', 'total_items']df_fe = df_fe.merge(agg_cliente, on='cliente_id', how='left')print("\n✅ Features de agregación:")print(df_fe[['cliente_id', 'total_gastado', 'gasto_promedio', 'num_compras']])# 4. Binning (edad en grupos)print("\n" + "="*50)print("🔧 4. BINNING (Edad en grupos)")print("="*50)df_fe['grupo_edad'] = pd.cut(df_fe['edad_cliente'],                               bins=[0, 18, 35, 65, 100],                               labels=['Joven', 'Adulto', 'Senior', 'Anciano'])print("\n✅ Binning de edad:")print(df_fe[['edad_cliente', 'grupo_edad']])# 5. Transformaciones matemáticasprint("\n" + "="*50)print("🔧 5. TRANSFORMACIONES MATEMÁTICAS")print("="*50)df_fe['log_precio'] = np.log1p(df_fe['precio'])  # log(1+x) para evitar log(0)df_fe['sqrt_cantidad'] = np.sqrt(df_fe['cantidad'])print("\n✅ Transformaciones:")print(df_fe[['precio', 'log_precio', 'cantidad', 'sqrt_cantidad']])# Resumen finalprint("\n" + "="*50)print("🏆 DATASET CON TODAS LAS FEATURES")print("="*50)print(f"\n📊 Features originales: 6")print(f"🎉 Features nuevas: {len(df_fe.columns) - 6}")print(f"🔢 Total features: {len(df_fe.columns)}")print("\nColumnas:")for i, col in enumerate(df_fe.columns, 1):    print(f"  {i}. {col}")

In [0]:
# DBTITLE 1,## Parte 4: Selección de Features# MAGIC %md# MAGIC ## Parte 4: Selección de Features# MAGIC# MAGIC ### ⚠️ El Problema de Muchas Features# MAGIC# MAGIC **"Curse of Dimensionality"** - Demasiadas features:# MAGIC * ❌ Overfitting# MAGIC * ❌ Mayor tiempo de entrenamiento# MAGIC * ❌ Modelos difíciles de interpretar# MAGIC * ❌ Ruido en los datos# MAGIC# MAGIC ### Métodos de Selección# MAGIC# MAGIC #### 1️⃣ **Correlación**# MAGIC# MAGIC **Eliminar features altamente correlacionadas:**# MAGIC * Si $\text{corr}(X_1, X_2) > 0.9$ → Eliminar una# MAGIC * Evita redundancia# MAGIC# MAGIC **Correlación de Pearson:**# MAGIC $$r = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^{n}(x_i - \bar{x})^2}\sqrt{\sum_{i=1}^{n}(y_i - \bar{y})^2}}$$# MAGIC# MAGIC $r \in [-1, 1]$:# MAGIC * $r = 1$: Correlación positiva perfecta# MAGIC * $r = 0$: Sin correlación# MAGIC * $r = -1$: Correlación negativa perfecta# MAGIC# MAGIC #### 2️⃣ **Feature Importance**# MAGIC# MAGIC Modelos basados en árboles (Random Forest, XGBoost) proveen importancia de features:# MAGIC * **Gini Importance**: Cuánto reduce la impureza# MAGIC * **Permutation Importance**: Caída en accuracy al permutar feature# MAGIC# MAGIC #### 3️⃣ **Recursive Feature Elimination (RFE)**# MAGIC# MAGIC 1. Entrenar modelo con todas las features# MAGIC 2. Rankear features por importancia# MAGIC 3. Eliminar la menos importante# MAGIC 4. Repetir hasta N features# MAGIC# MAGIC #### 4️⃣ **Variance Threshold**# MAGIC# MAGIC Eliminar features con baja varianza:# MAGIC * Si $\text{Var}(X) < \text{threshold}$ → Eliminar# MAGIC * Features constantes o casi-constantes no aportan

In [0]:
# DBTITLE 1,Ejemplo: Selección de featuresfrom sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classiffrom sklearn.ensemble import RandomForestClassifierimport seaborn as snsimport matplotlib.pyplot as plt# Crear dataset para clasificaciónnp.random.seed(42)n = 100df_selection = pd.DataFrame({    'feature1': np.random.randn(n),    'feature2': np.random.randn(n) * 2,    'feature3': np.random.randn(n) * 0.1,  # Baja varianza    'feature4': np.ones(n),  # Constante    'feature5': np.random.randn(n),})# Feature 2 es altamente correlacionada con feature 1df_selection['feature2'] = df_selection['feature1'] * 0.9 + np.random.randn(n) * 0.1# Targetdf_selection['target'] = (df_selection['feature1'] + df_selection['feature5'] > 0).astype(int)print("┌──────────────────────────────────────────────────┐")print("│         SELECCIÓN DE FEATURES - EJEMPLOS           │")print("└──────────────────────────────────────────────────┘")# 1. Variance Thresholdprint("\n" + "="*50)print("🔧 1. VARIANCE THRESHOLD")print("="*50)features = df_selection.drop('target', axis=1)print("\n📊 Varianza de cada feature:")for col in features.columns:    print(f"  {col}: {features[col].var():.4f}")var_threshold = VarianceThreshold(threshold=0.05)features_high_var = var_threshold.fit_transform(features)selected_features = features.columns[var_threshold.get_support()]print(f"\n✅ Features seleccionadas (varianza > 0.05): {list(selected_features)}")print(f"❌ Features eliminadas: {list(set(features.columns) - set(selected_features))}")# 2. Correlaciónprint("\n" + "="*50)print("🔧 2. MATRIZ DE CORRELACIÓN")print("="*50)corr_matrix = features.corr()print("\n📊 Matriz de correlación:")print(corr_matrix.round(2))print("\n⚠️  Pares altamente correlacionados (|r| > 0.8):")for i in range(len(corr_matrix.columns)):    for j in range(i+1, len(corr_matrix.columns)):        if abs(corr_matrix.iloc[i, j]) > 0.8:            print(f"  {corr_matrix.columns[i]} <-> {corr_matrix.columns[j]}: {corr_matrix.iloc[i, j]:.3f}")# 3. Feature Importance con Random Forestprint("\n" + "="*50)print("🔧 3. FEATURE IMPORTANCE (Random Forest)")print("="*50)X = df_selection.drop('target', axis=1)y = df_selection['target']rf = RandomForestClassifier(n_estimators=100, random_state=42)rf.fit(X, y)importances = pd.DataFrame({    'feature': X.columns,    'importance': rf.feature_importances_}).sort_values('importance', ascending=False)print("\n🏆 Feature Importance:")print(importances.to_string(index=False))print(f"\n✅ Top 3 features más importantes:")for i, row in importances.head(3).iterrows():    print(f"  {row['feature']}: {row['importance']:.4f}")print("\n\n📝 CONCLUSIONES:")print("="*50)print("✅ Eliminar features con baja varianza (feature3, feature4)")print("✅ Eliminar una de las features correlacionadas (feature1 o feature2)")print("✅ Priorizar features con alta importancia (feature1, feature5)")print("\n🎯 Features finales recomendadas: feature1, feature5")

In [0]:
# DBTITLE 1,## 📝 Conclusiones y Mejores Prácticas# MAGIC %md# MAGIC ## 📝 Conclusiones y Mejores Prácticas# MAGIC# MAGIC ### 🏆 Key Takeaways# MAGIC# MAGIC 1. **El 80% del trabajo en ML es preparación de datos**# MAGIC    - Limpieza, transformaciones, feature engineering# MAGIC    - Modelos mediocres con buenos datos > Modelos sofisticados con datos sucios# MAGIC# MAGIC 2. **No existe una "receta única"**# MAGIC    - Depende del problema, datos y algoritmo# MAGIC    - Experimentar con diferentes técnicas# MAGIC# MAGIC 3. **Feature Engineering es un arte**# MAGIC    - Requiere domain knowledge# MAGIC    - Creatividad y experimentación# MAGIC    - "Features beat algorithms"# MAGIC# MAGIC 4. **Menos es más**# MAGIC    - Eliminar features redundantes o irrelevantes# MAGIC    - Evitar overfitting# MAGIC    - Modelos más interpretables# MAGIC# MAGIC ---# MAGIC# MAGIC ### ✅ Checklist de Preprocesamiento# MAGIC# MAGIC **Antes de entrenar cualquier modelo:**# MAGIC# MAGIC ☐ **Explorar datos**# MAGIC   - Estadísticas descriptivas# MAGIC   - Visualizaciones# MAGIC   - Identificar problemas# MAGIC# MAGIC ☐ **Limpieza**# MAGIC   - Valores faltantes → Imputar o eliminar# MAGIC   - Duplicados → Eliminar# MAGIC   - Outliers → Manejar apropiadamente# MAGIC# MAGIC ☐ **Transformaciones**# MAGIC   - Encoding categóricas → Label, One-Hot, Target# MAGIC   - Scaling numéricas → StandardScaler, MinMaxScaler# MAGIC   - Transformaciones matemáticas si es necesario# MAGIC# MAGIC ☐ **Feature Engineering**# MAGIC   - Crear features de dominio# MAGIC   - Interacciones# MAGIC   - Agregaciones# MAGIC# MAGIC ☐ **Selección de Features**# MAGIC   - Eliminar baja varianza# MAGIC   - Eliminar alta correlación# MAGIC   - Feature importance# MAGIC# MAGIC ☐ **Validar**# MAGIC   - Train/test split ANTES de preprocesar# MAGIC   - Fit en train, transform en test# MAGIC   - Evitar data leakage# MAGIC# MAGIC ---# MAGIC# MAGIC ### ⚠️ Errores Comunes a Evitar# MAGIC# MAGIC 1. **Data Leakage**# MAGIC    ❌ Fit/transform en todo el dataset# MAGIC    ✅ Fit en train, transform en train y test por separado# MAGIC# MAGIC 2. **Imputar antes de split**# MAGIC    ❌ Imputar con media de todo el dataset# MAGIC    ✅ Split primero, luego imputar con media del train# MAGIC# MAGIC 3. **Eliminar outliers sin justificación**# MAGIC    ❌ Eliminar automáticamente todos los outliers# MAGIC    ✅ Analizar si son errores o valores legítimos# MAGIC# MAGIC 4. **One-Hot Encoding de alta cardinalidad**# MAGIC    ❌ 100+ categorías → 100+ columnas# MAGIC    ✅ Usar Target Encoding o Frequency Encoding# MAGIC# MAGIC 5. **Scaling innecesario**# MAGIC    ❌ Scaling en árboles de decisión (no necesario)# MAGIC    ✅ Scaling solo para algoritmos sensibles (SVM, KNN, NN)# MAGIC# MAGIC ---# MAGIC# MAGIC ### 📚 Recursos Adicionales# MAGIC# MAGIC **Libros:**# MAGIC * **"Feature Engineering for Machine Learning"** - Alice Zheng# MAGIC * **"Python for Data Analysis"** - Wes McKinney# MAGIC# MAGIC **Cursos:**# MAGIC * Kaggle Learn - Feature Engineering# MAGIC * Fast.ai - Practical Deep Learning# MAGIC# MAGIC **Herramientas:**# MAGIC * **Pandas Profiling**: EDA automático# MAGIC * **featuretools**: Feature engineering automatizado# MAGIC * **category_encoders**: Múltiples métodos de encoding# MAGIC# MAGIC ---# MAGIC# MAGIC ### 🚀 Próximos Pasos# MAGIC# MAGIC **En el siguiente notebook** (`04_Evaluacion_y_Validacion.ipynb`):# MAGIC * Métricas de evaluación# MAGIC * Validación cruzada# MAGIC * Overfitting y Underfitting# MAGIC * Bias-Variance Tradeoff# MAGIC# MAGIC **Luego:**# MAGIC * Aprendizaje Supervisado (algoritmos específicos)# MAGIC * Aprendizaje No Supervisado# MAGIC * Aprendizaje por Refuerzo# MAGIC * AutoML y MLOps# MAGIC# MAGIC ---# MAGIC# MAGIC ## 🎉 ¡Felicidades!# MAGIC# MAGIC Ahora dominas las técnicas fundamentales de **Preprocesamiento y Feature Engineering**.# MAGIC# MAGIC Estos conocimientos son la base para **cualquier proyecto de Machine Learning exitoso**. 🚀